In [ ]:
### Imports

In [59]:
import pandas as pd

## Chargement des fichiers CSV 

In [66]:
# Charger les fichiers CSV
df_large = pd.read_csv('../data/raw/bank-full.csv', sep=';')
df_small = pd.read_csv('../data/raw/bank.csv', sep=';')

## Exploration des datasets

### Exploration df_large

#### Recherche de doublons

In [16]:
# Vérifier le nombre de doublons dans le DataFrame
num_duplicates = df_large.duplicated().sum()

# Afficher le nombre de doublons
print(f"Nombre de doublons dans le DataFrame : {num_duplicates}")

Nombre de doublons dans le DataFrame : 0


#### **Analyse Structurelle : Métadonnées et Intégrité du Dataset**

In [12]:
info_large = df_large.info()
print(info_large)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB
None


#### **1. Complétude des Données**
  **Aucune valeur manquante détectée**  
- Toutes les colonnes présentent un `Non-Null Count` égal au nombre total d'entrées (`45 211`)  
- **Implication** :  
  - Aucun traitement de imputation nécessaire  
  - Données immédiatement exploitables pour l'analyse  

#### **2. Cohérence des Types de Données**  
  **Typage conforme aux attentes**  
- **Variables numériques** :  
  - `age`, `balance`, `duration` correctement en `int64`/`float64`  
- **Variables catégorielles** :  
  - `job`, `education`, etc. bien typées en `object`  
- **Variables binaires** :  
  - `default`, `housing` encodées en texte (`object`)  

#### **Analyse Exploratoire Initiale : Statistiques Descriptives**

In [11]:
description_large = df_large.describe()
print(description_large)

                age        balance           day      duration      campaign  \
count  45211.000000   45211.000000  45211.000000  45211.000000  45211.000000   
mean      40.936210    1362.272058     15.806419    258.163080      2.763841   
std       10.618762    3044.765829      8.322476    257.527812      3.098021   
min       18.000000   -8019.000000      1.000000      0.000000      1.000000   
25%       33.000000      72.000000      8.000000    103.000000      1.000000   
50%       39.000000     448.000000     16.000000    180.000000      2.000000   
75%       48.000000    1428.000000     21.000000    319.000000      3.000000   
max       95.000000  102127.000000     31.000000   4918.000000     63.000000   

              pdays      previous  
count  45211.000000  45211.000000  
mean      40.197828      0.580323  
std      100.128746      2.303441  
min       -1.000000      0.000000  
25%       -1.000000      0.000000  
50%       -1.000000      0.000000  
75%       -1.000000      0.

### Analyse des principales observations

#### 1. Colonne "campaign" (nombre de contacts)
- **Moyenne** : 3 contacts
- **Écart-type** : 3 (forte dispersion)
- **Distribution** :
  - 75% des clients ont ≤3 contacts
  - Présence possible de valeurs extrêmes

#### 2. Colonne "duration" (durée d'appel)
- **Problèmes** :
  - Valeurs min=0 (non plausibles)
  - Valeurs max trop élevées
- **Recommandation** :
  - Filtrer les durées=0
  - Examiner les outliers

#### 3. Colonne "contact"
- Peu informative dans sa forme actuelle
- Possible suppression

#### 4. Colonne "previous"
- Valeur max=275 (outlier probable)
- À investiguer

#### 5. Colonnes "balance" et "age"
- Distributions cohérentes
- Pas d'anomalie détectée

### Analyse colonne par colonne

#### "job"

In [46]:
print(df_large['job'].value_counts())

job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
unknown           288
Name: count, dtype: int64


Observations: 288 valeurs inconnus

#### "education"

In [47]:
print(df_large['education'].value_counts())

education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64


Observations: 1857 valeurs inconnus

#### 'contact'

In [48]:
print(df_large['contact'].value_counts())

contact
cellular     29285
unknown      13020
telephone     2906
Name: count, dtype: int64


In [ ]:
Observations: 13020 valeurs inconnus

#### 'day'

In [51]:
print(df_large['day'].value_counts())

day
20    2752
18    2308
21    2026
17    1939
6     1932
5     1910
14    1848
8     1842
28    1830
7     1817
19    1757
29    1745
15    1703
12    1603
13    1585
30    1566
9     1561
11    1479
4     1445
16    1415
2     1293
27    1121
3     1079
26    1035
23     939
22     905
25     840
31     643
10     524
24     447
1      322
Name: count, dtype: int64


In [ ]:
Segmentation en début de mois et fin de mois

In [69]:
# Compter les entrées entre 1 et 15 inclus
count_1_15 = len(df_large[(df_large['day'] >= 1) & (df_large['day'] <= 15)])

# Compter les entrées entre 16 et 31 inclus
count_16_31 = len(df_large[(df_large['day'] >= 16) & (df_large['day'] <= 31)])

print(f"Nombre d'entrées pour day 1-15 : {count_1_15}")
print(f"Nombre d'entrées pour day 16-31 : {count_16_31}")

Nombre d'entrées pour day 1-15 : 21943
Nombre d'entrées pour day 16-31 : 23268


observations: Possibilité de partager en deux parties quasi-éguales 

#### 'month'

In [52]:
print(df_large['month'].value_counts())

month
may    13766
jul     6895
aug     6247
jun     5341
nov     3970
apr     2932
feb     2649
jan     1403
oct      738
sep      579
mar      477
dec      214
Name: count, dtype: int64


In [ ]:
Observations: Mauvaise répartition des entrées en fonction des mois

#### 'pdays'

In [54]:
print(df_large['pdays'].value_counts())

pdays
-1      36954
 182      167
 92       147
 91       126
 183      126
        ...  
 449        1
 452        1
 648        1
 595        1
 530        1
Name: count, Length: 559, dtype: int64


Nombre d'entrée au dessus de 365 jours

In [56]:
count_above_10 = (df_large['pdays'] > 365).sum()
print(count_above_10)

643


Observations: 
- Une grande partie des entrées sont un premier contact.
- Certaines valeurs trop hautes sont unique (643 supérieures à 1 an)

#### 'previous'

In [57]:
print(df_large['previous'].value_counts())

previous
0      36954
1       2772
2       2106
3       1142
4        714
5        459
6        277
7        205
8        129
9         92
10        67
11        65
12        44
13        38
15        20
14        19
17        15
16        13
19        11
20         8
23         8
18         6
22         6
24         5
27         5
21         4
29         4
25         4
30         3
38         2
37         2
26         2
28         2
51         1
275        1
58         1
32         1
40         1
55         1
35         1
41         1
Name: count, dtype: int64


Nombre d'entrées avec une valeurs au dessus de 30

In [71]:
count_above_30 = (df_large['previous'] > 19).sum()
print(count_above_30)

63


Observations: 
- On retrouve le même nombre d'entrées en premier contact
- Certaines valeurs hautes n'ont pas d'occurences suffisantes 

#### 'campaign'

In [72]:
print(df_large['campaign'].value_counts())

campaign
1     17544
2     12505
3      5521
4      3522
5      1764
6      1291
7       735
8       540
9       327
10      266
11      201
12      155
13      133
14       93
15       84
16       79
17       69
18       51
19       44
20       43
21       35
22       23
25       22
23       22
24       20
29       16
28       16
26       13
31       12
27       10
32        9
30        8
33        6
34        5
36        4
35        4
43        3
38        3
37        2
50        2
41        2
46        1
58        1
55        1
63        1
51        1
39        1
44        1
Name: count, dtype: int64


In [45]:
count_above_10 = (df_large['campaign'] > 10).sum()
print(count_above_10)

1196


Observations :
- 1196 entrées avec un nombres d'appels supérieur à 10 pour cette campagne

#### 'poutcome'

In [73]:
print(df_large['poutcome'].value_counts())

poutcome
unknown    36959
failure     4901
other       1840
success     1511
Name: count, dtype: int64


Observations: 
- Une grande majorité d'entrées avec valeur inconnu

In [ ]:
#### 'duration'

In [74]:
print(df_large['duration'].value_counts())

duration
124     188
90      184
89      177
104     175
122     175
       ... 
1833      1
1545      1
1352      1
1342      1
1556      1
Name: count, Length: 1573, dtype: int64


In [79]:
count_above_1000 = (df_large['duration'] >900 ).sum()
print(count_above_1000)

1418


Observations: 
- Répartitions des valeurs trés étérogènes
- 3 entrées à 0 => Incohérence
- 1418 entrées au dessus de 16 minutes de communication

In [77]:
count_0 = (df_large['duration'] == 0 ).sum()
print(count_0)

3


In [ ]:
#### Conclusion de nos observations de ce dataset